# Chapter 6: Workflow Before Intelligence
## The Three Structural Primitives

This notebook demonstrates the three workflow primitives from the chapter using a simulated clinical documentation agent.The agent fetches patient data from multiple sources and makes a diagnostic coding recommendation.

**The central argument:** The most consequential design decision is not which model powers the system - it is how the workflow is *structured*. Get it wrong and intelligence becomes a liability: a faster path to a more confident error.


# Setup — Simulated Data Sources

Each function below represents a real data source the agent would query (labs, notes, medications, etc.).
We add `time.sleep()` to simulate realistic network latency — this is what makes the timing comparisons meaningful.

**One source is intentionally broken:** `fetch_lab_results_BROKEN` returns a partial result
but marks its status as `"final"`. This mimics the real failure case described in the chapter
(a lab system under high load returning incomplete data with a misleading status flag).

In [15]:
import time
import asyncio
import random
from concurrent.futures import ThreadPoolExecutor

In [16]:
def fetch_progress_note(patient_id):
    time.sleep(0.8)
    return {"source": "progress_note", "text": "Patient recovering well post knee surgery."}

def fetch_lab_results(patient_id):
    time.sleep(0.6)
    return {"source": "labs", "troponin": "0.01 ng/mL", "status": "final"}

def fetch_medication_list(patient_id):
    time.sleep(0.7)
    return {"source": "medications", "current": ["ibuprofen 400mg", "hydromorphone 2mg"]}

def fetch_allergy_profile(patient_id):
    time.sleep(0.5)
    return {"source": "allergies", "allergies": ["penicillin"]}

def fetch_cardiology_note(patient_id):
    time.sleep(0.9)
    return {"source": "cardiology_note", "text": "No acute cardiac findings."}

# A broken lab source that returns partial data but marks it as "final"
def fetch_lab_results_BROKEN(patient_id):
    time.sleep(0.6)
    return {"source": "labs", "troponin": "0.01 ng/mL", "status": "final", "_partial": True}

## Primitive 1 — Sequential Processing

**Design decision:** Task B does not begin until Task A has fully completed and its output is available.

**When to use it:** When there is a *logical dependency* between steps — i.e., the output of step A
is a required input to step B. In clinical documentation, you cannot propose a diagnostic code
before you have read the physician's progress note. Skipping that order is not just slow — it is a compliance violation.

**The cost:** Latency stacks up. If each retrieval takes 0.8s, 0.6s, and 0.7s, you wait the full 2.1s
before any reasoning begins.

**The 2015 overdose case** (described in the chapter opening) is what happens when two steps that
*should* be sequential are allowed to run in parallel: verify then record became verify *and* record simultaneously,
and a patient received a 10x overdose because both systems assumed the other hadn't acted yet.

In [17]:
print("=" * 60)
print("1. SEQUENTIAL PROCESSING")
print("=" * 60)
print("Rule: Step B cannot start until Step A finishes.\n")

patient_id = "P-001"
start = time.time()

print("  [1/3] Fetching progress note...")
note = fetch_progress_note(patient_id)

print("  [2/3] Fetching lab results (depends on note being present)...")
labs = fetch_lab_results(patient_id)

print("  [3/3] Fetching medication list (depends on labs being reviewed)...")
meds = fetch_medication_list(patient_id)

elapsed = time.time() - start
print(f"\n  ✅ All data fetched sequentially.")
print(f"  ⏱  Total time: {elapsed:.2f}s  (≈ 0.8 + 0.6 + 0.7 = 2.1s expected)\n")

1. SEQUENTIAL PROCESSING
Rule: Step B cannot start until Step A finishes.

  [1/3] Fetching progress note...
  [2/3] Fetching lab results (depends on note being present)...
  [3/3] Fetching medication list (depends on labs being reviewed)...

  ✅ All data fetched sequentially.
  ⏱  Total time: 2.10s  (≈ 0.8 + 0.6 + 0.7 = 2.1s expected)



# Primitive 2 — Parallel Processing

**Design decision:** Tasks with *no logical dependency on each other* are dispatched simultaneously.
The agent waits for all of them to resolve before any reasoning begins.

**The critical constraint:** The engineer — not the agent — decides at *design time* what can be parallelized.
The agent does not decide this at runtime. This distinction matters: if the agent were choosing dynamically,
a reasoning error about independence could silently introduce a dependency violation.

**When to use it:** When retrievals are genuinely independent. A patient's allergy profile does not change
based on what the troponin shows. The cardiology note exists regardless of the medication list. These are safe to run together.

**The payoff:** Instead of waiting for the sum of all latencies (~2.7s for four sources),
you wait for the *slowest* one (~0.9s). The model didn't change. The data didn't change.
The workflow structure alone produced a 3x speedup.

In [18]:
print("=" * 60)
print("2. PARALLEL PROCESSING")
print("=" * 60)
print("Rule: Independent retrievals run at the same time.\n")

start = time.time()

with ThreadPoolExecutor() as pool:
    f_labs     = pool.submit(fetch_lab_results,     patient_id)
    f_meds     = pool.submit(fetch_medication_list, patient_id)
    f_allergies= pool.submit(fetch_allergy_profile, patient_id)
    f_cardio   = pool.submit(fetch_cardiology_note, patient_id)

    labs      = f_labs.result()
    meds      = f_meds.result()
    allergies = f_allergies.result()
    cardio    = f_cardio.result()

elapsed = time.time() - start
print(f"  ✅ All 4 sources fetched concurrently.")
print(f"  ⏱  Total time: {elapsed:.2f}s  (≈ max(0.6,0.7,0.5,0.9) = 0.9s expected)\n")
print("  Sources retrieved:", [r["source"] for r in [labs, meds, allergies, cardio]])
print()

2. PARALLEL PROCESSING
Rule: Independent retrievals run at the same time.

  ✅ All 4 sources fetched concurrently.
  ⏱  Total time: 0.91s  (≈ max(0.6,0.7,0.5,0.9) = 0.9s expected)

  Sources retrieved: ['labs', 'medications', 'allergies', 'cardiology_note']



## Primitive 3 — Suspend / Resume (Human Approval Gate)

**Design decision:** Certain decisions are structurally prohibited from being made by the agent alone.
When the agent reaches one of these points, execution stops and a human must act before it can continue.

**This is not a temporary scaffold.** The chapter is explicit: the suspend/resume primitive is a
*permanent* architectural feature of systems where errors carry irreversible consequences and
accountability must be traceable to a person. It is not removed when the model gets smarter.

**What triggers a suspend here:** A "high-confidence divergence" — the agent's proposed code
differs from the physician's documented language, and the stakes of being wrong are categorized as high.

**Two paths are shown below** to make both outcomes visible without blocking the notebook.
In production, this would be a UI prompt surfaced to the attending physician with a configurable timeout.

In [19]:
print("=" * 60)
print("3. SUSPEND / RESUME  (Human Approval Gate)")
print("=" * 60)
print("Rule: When the agent hits an ambiguous decision, it STOPS")
print("      and waits for a human before continuing.\n")

def propose_diagnostic_code(labs, note):
    # Simulate a high-confidence divergence
    agent_code   = "I21.9"   # Acute MI (what the agent thinks based on troponin trend)
    physician_doc = "Z96.641" # Knee replacement status (what the note says)
    return agent_code, physician_doc

def suspend_for_human_review(agent_code, physician_code):
    print(f"  ⚠️  DIVERGENCE DETECTED")
    print(f"      Agent proposed:      {agent_code}  (Acute MI)")
    print(f"      Physician documented: {physician_code} (Knee replacement)")
    print(f"\n  🛑 Workflow SUSPENDED. Awaiting physician input...")
    decision = input("\n  Physician: approve agent code (a) or keep documented code (k)? [a/k]: ").strip().lower()
    return agent_code if decision == "a" else physician_code

agent_code, physician_code = propose_diagnostic_code(labs, note)

if agent_code != physician_code:
    final_code = suspend_for_human_review(agent_code, physician_code)
else:
    final_code = agent_code

print(f"\n  ✅ Workflow RESUMED. Final code submitted: {final_code}\n")

3. SUSPEND / RESUME  (Human Approval Gate)
Rule: When the agent hits an ambiguous decision, it STOPS
      and waits for a human before continuing.

  ⚠️  DIVERGENCE DETECTED
      Agent proposed:      I21.9  (Acute MI)
      Physician documented: Z96.641 (Knee replacement)

  🛑 Workflow SUSPENDED. Awaiting physician input...

  ✅ Workflow RESUMED. Final code submitted: I21.9



## Deliberate Failure — Premature Finalization

**The failure mode:** A parallel retrieval architecture fetches data from multiple sources concurrently
and then merges them into a single context for the model to reason from. The merge step is designed to
proceed as soon as all retrievals return a non-null result.

**The problem:** Non-null is not the same as complete. The lab system, under high load, returns
a partial result with `status: "final"` — a known bug in the source system. The agent has no way
to know the data is incomplete. It proceeds, reasons correctly over the wrong inputs, and submits a billing code.

**This is exactly the JAMIA near-miss case from the chapter:** 11 documented events over four months,
three of which involved clinically significant miscoding. One patient's record was finalized with a
sepsis code when the culture result that would have confirmed or refuted it had not yet been incorporated.

**The fix — a data completeness gate:** Before any retrieval result is passed to the reasoning layer,
validate not just that something was returned, but that its *metadata* (status flags, timestamps,
source confidence indicators) is consistent with a finalized, complete record.
Partial or anomalous results trigger a suspend, not a proceed.

In [20]:
print("=" * 60)
print("DELIBERATE FAILURE — Premature Finalization")
print("=" * 60)
print("What breaks: the merge step treats a PARTIAL lab result")
print("as complete because status='final' (the broken lab system).\n")

def merge_and_finalize_UNSAFE(results):
    """No completeness check — proceeds if any result is non-null."""
    for r in results:
        if r is None:
            raise ValueError("Null result — aborting.")
    print("  Merge step: all results non-null → proceeding ✓")
    return "Diagnostic code finalized based on merged context."

def merge_and_finalize_SAFE(results):
    """Completeness gate — checks metadata before proceeding."""
    for r in results:
        if r is None:
            raise ValueError("Null result — aborting.")
        if r.get("_partial"):
            print(f"  🛑 Data completeness gate FIRED on: {r['source']}")
            print(f"     Partial data detected — workflow SUSPENDED.")
            return None
    print("  Merge step: all results complete → proceeding ✓")
    return "Diagnostic code finalized based on merged context."

DELIBERATE FAILURE — Premature Finalization
What breaks: the merge step treats a PARTIAL lab result
as complete because status='final' (the broken lab system).



## Simulate fetching with the broken lab source

In [21]:
print("--- UNSAFE (no completeness gate) ---")
broken_labs = fetch_lab_results_BROKEN(patient_id)
results = [broken_labs, meds, allergies]
outcome = merge_and_finalize_UNSAFE(results)
print(f"  Result: {outcome}")
print(f"  ❌ ERROR: Billing submitted on incomplete lab data!\n")

print("--- SAFE (with completeness gate) ---")
broken_labs = fetch_lab_results_BROKEN(patient_id)
results = [broken_labs, meds, allergies]
outcome = merge_and_finalize_SAFE(results)
if outcome is None:
    print(f"  ✅ Correct: agent did not finalize. Awaiting complete lab results.")

--- UNSAFE (no completeness gate) ---
  Merge step: all results non-null → proceeding ✓
  Result: Diagnostic code finalized based on merged context.
  ❌ ERROR: Billing submitted on incomplete lab data!

--- SAFE (with completeness gate) ---
  🛑 Data completeness gate FIRED on: labs
     Partial data detected — workflow SUSPENDED.
  ✅ Correct: agent did not finalize. Awaiting complete lab results.
